In [ ]:
%pip install -U chromadb langchain langchain-classic langchain-groq langchain-community \
  langchain-chroma langchain-text-splitters sentence-transformers langchain-unstructured langchain-huggingface
%pip install -U "unstructured[pdf]"

In [ ]:
!apt-get install poppler

In [ ]:
!apt-get update && apt-get install -y poppler-utils tesseract-ocr

import os
import sys
import site
from importlib import reload

# Refresh site packages
site.main()

try:
    # Try the standard import first
    from langchain.chains import RetrievalQA
except (ImportError, ModuleNotFoundError):
    print("Module not found, performing a targeted install and reload...")
    !pip install -U langchain langchain-community langchain-classic
    import site
    reload(site)
    import langchain
    reload(langchain)
    # Try importing from langchain_classic as a fallback if legacy structure is needed
    try:
        from langchain.chains import RetrievalQA
    except:
        from langchain_classic.chains import RetrievalQA

from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

print("Imports successful.")

In [ ]:
!pip install --upgrade "unstructured[local-inference]"

In [ ]:
import os
# Re-setting the API key to ensure it is in the environment
os.environ["GROQ_API_KEY"] = "your-api-key-here"
print("API Key set in environment.")

In [ ]:
import requests
url = "https://courses.cs.umbc.edu/671/fall12/notes/python/04_python_functions.pdf"

response = requests.get(url)

In [ ]:
# save file to the local directory
with open("python_inbuildfunctions.pdf", "wb") as f:
    f.write(response.content)

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader("python_inbuildfunctions.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")
documents  # this will be list of document objects

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=50)

In [ ]:
text = text_splitter.split_documents(documents)

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
persist_directory = 'vector_db'

In [ ]:
from langchain_community.vectorstores.utils import filter_complex_metadata

# Filter out complex metadata (like nested dicts) that Chroma cannot handle
filtered_text = filter_complex_metadata(text)

# Now create the vector store with cleaned metadata
vectordb = Chroma.from_documents(
    documents=filtered_text,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("Vector database created successfully.")

In [ ]:
retriver = vectordb.as_retriever()

In [ ]:
import os
from langchain_groq import ChatGroq

api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found in environment. Please ensure cell vTk6Gp2PHMTy was executed.")

# Explicitly passing the API key string to the constructor
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=api_key
)

print("LLM initialized successfully.")

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriver,
    return_source_documents=True
)

In [ ]:
query = 'Explain about filter,map, reduce'
response = qa_chain.invoke({"query": query})
print(response)

In [ ]:
print(response['result'])